# AssureX Warranty Claims — Feature Engineering & Exploratory Data Analysis

**Author:** AssureX Machine Learning & Claims Analytics Team  
**Target Objective:** Multi-Class Warranty Claim Classification (`Valid Claim`, `Invalid Claim`, `Manual Review`)  

This Google Colab / Jupyter notebook implements an end-to-end feature engineering and exploratory data analysis (EDA) pipeline:
1. **Data Ingestion:** Loads `train.csv`, `val.csv`, and `test.csv`.
2. **Derived Feature Engineering:** Computes business domain indicators (`product_age_days`, `remaining_warranty_days`, `missing_document_count`, `days_to_reporting_deadline`, `repair_count`, and `has_any_contradiction`).
3. **Exploratory Visualizations:** Plots target class distribution, feature correlation heatmap vs. class label, and product age distributions by class.
4. **Categorical Encoding:** Applies One-Hot Encoding for low-cardinality nominal variables and Target/Frequency Encoding for high-cardinality variables with rigorous data-leakage prevention (fitted strictly on the training partition).
5. **Artifact Export:** Saves the processed feature matrices as `train_features.csv`, `val_features.csv`, and `test_features.csv`.

In [ ]:
# ==============================================================================
# Step 0: Environment Setup & Core Dependencies
# ==============================================================================
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting defaults
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Environment initialized successfully. Pandas version:", pd.__version__)

---  
## Step 1: Data Ingestion (`train.csv`, `val.csv`, `test.csv`)

We load the three stratified partitions. The loader automatically checks current directory, `./dataset/`, and standard Colab paths (`/content/`).

In [ ]:
def locate_data_file(filename: str) -> str:
    """Search multiple common paths for split CSV files."""
    candidate_locations = [
        filename,
        os.path.join("dataset", filename),
        os.path.join("/content", filename),
        os.path.join("/content", "dataset", filename),
        os.path.join("..", "dataset", filename)
    ]
    for loc in candidate_locations:
        if os.path.exists(loc):
            return loc
    raise FileNotFoundError(f"Unable to find {filename} in candidate paths: {candidate_locations}")

train_raw = pd.read_csv(locate_data_file("train.csv"))
val_raw = pd.read_csv(locate_data_file("val.csv"))
test_raw = pd.read_csv(locate_data_file("test.csv"))

print(f"[+] Train split loaded:      {train_raw.shape[0]} rows, {train_raw.shape[1]} columns")
print(f"[+] Validation split loaded: {val_raw.shape[0]} rows, {val_raw.shape[1]} columns")
print(f"[+] Test split loaded:       {test_raw.shape[0]} rows, {test_raw.shape[1]} columns")

---  
## Step 2: Derived Feature Engineering

We extract six domain-informed derived features from the temporal sequences, document checklists, service logs, and fraud signals:

1. **`product_age_days`**: Elapsed days from the original retail purchase date to the claim submission date (`claim_submission_date - purchase_date`). Measures overall equipment lifespan.
2. **`remaining_warranty_days`**: Difference between `warranty_expiry_date` and `fault_occurrence_date`. Positive values indicate active coverage at the moment of breakdown; negative values quantify days elapsed since warranty expiration.
3. **`missing_document_count`**: Integer sum ($0$ to $4$) of missing compliance items among `has_receipt`, `has_warranty_card`, `has_product_image`, and `has_serial_evidence`.
4. **`days_to_reporting_deadline`**: Statutory policy compliance metric. Calculated as the 7-day reporting deadline minus actual submission date: `(fault_occurrence_date + 7 days) - claim_submission_date`. Positive indicates prompt reporting within the 7-day window; negative indicates late submission.
5. **`repair_count`**: Integer count ($0$ to $3$) of prior repair interventions extracted from the structured `repair_history` string.
6. **`has_any_contradiction`**: Boolean flag flagging internal data integrity violations, including chronological impossibilities (e.g. claim filed before fault, fault occurring before purchase), receipt serial mismatches when receipt is attached, replacement exhausted violations (`prior_replacement == True`), or contradictions between narrative descriptions and damage categories.

In [ ]:
def compute_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Computes domain-specific derived features for claims classification.
    """
    df = df.copy()

    # Parse date strings to pandas datetime
    dt_purchase = pd.to_datetime(df['purchase_date'], errors='coerce')
    dt_fault = pd.to_datetime(df['fault_occurrence_date'], errors='coerce')
    dt_expiry = pd.to_datetime(df['warranty_expiry_date'], errors='coerce')
    dt_claim = pd.to_datetime(df['claim_submission_date'], errors='coerce')

    # 1. product_age_days: purchase_date to claim_submission_date
    df['product_age_days'] = (dt_claim - dt_purchase).dt.days

    # 2. remaining_warranty_days: warranty_expiry_date - fault_occurrence_date
    df['remaining_warranty_days'] = (dt_expiry - dt_fault).dt.days

    # 3. missing_document_count: count of False among the 4 mandatory checklist flags
    doc_flags = ['has_receipt', 'has_warranty_card', 'has_product_image', 'has_serial_evidence']
    df['missing_document_count'] = 0
    for col in doc_flags:
        df['missing_document_count'] += (~df[col].astype(bool)).astype(int)

    # 4. days_to_reporting_deadline: (fault_date + 7 days) - claim_submission_date
    reporting_deadline = dt_fault + pd.to_timedelta(7, unit='D')
    df['days_to_reporting_deadline'] = (reporting_deadline - dt_claim).dt.days

    # 5. repair_count: integer extracted from repair_history
    df['repair_count'] = df['repair_history'].astype(str).str.extract(r'(\d+)')[0].fillna(0).astype(int)

    # 6. has_any_contradiction: boolean composite flag
    # - Date Chronology contradictions
    chronology_mismatch = (dt_claim < dt_fault) | (dt_fault < dt_purchase) | (dt_claim < dt_purchase)

    # - Serial number mismatch when receipt is present
    sn_unit = df['serial_number'].astype(str).str.strip().str.upper()
    sn_rcpt = df['serial_number_on_receipt'].astype(str).str.strip().str.upper()
    serial_mismatch = (
        df['has_receipt'].astype(bool) &
        (df['serial_number_on_receipt'].isna() | (sn_rcpt == 'NAN') | (sn_rcpt == '') | (sn_rcpt != sn_unit))
    )

    # - Duplicate replacement contradiction (claiming on voided serial)
    prior_replacement_flag = df['prior_replacement'].astype(bool)

    # - Condition vs text narrative contradiction
    desc_txt = df['fault_description'].astype(str).str.lower()
    type_txt = df['damage_type'].astype(str).str.lower()
    narrative_mismatch = (
        type_txt.str.contains('glitch|unspecified|wear', na=False) &
        desc_txt.str.contains('water|pool|spill|tea|drop|shatter|cracked', na=False)
    )

    df['has_any_contradiction'] = (
        chronology_mismatch | serial_mismatch | prior_replacement_flag | narrative_mismatch
    ).astype(bool)

    return df

# Apply transformations across splits
train_derived = compute_derived_features(train_raw)
val_derived = compute_derived_features(val_raw)
test_derived = compute_derived_features(test_raw)

print("[+] Derived features computed successfully across all splits!")
train_derived[['claim_id', 'product_age_days', 'remaining_warranty_days',
               'missing_document_count', 'days_to_reporting_deadline',
               'repair_count', 'has_any_contradiction']].head()

---  
## Step 3: Exploratory Data Analysis & Visualizations

We now visualize:
1. **Class Balance:** Verifying stratification across `Valid Claim`, `Invalid Claim`, and `Manual Review`.
2. **Correlation Heatmap:** Investigating linear relationships between numerical/derived features and target decisions.
3. **Boxplot of Product Age by Class:** Assessing device lifespan distributions across the three decision outcomes.

In [ ]:
# Plot 1: Target Class Balance in Training Set
plt.figure(figsize=(8, 5))
class_colors = {'Valid Claim': '#10B981', 'Invalid Claim': '#EF4444', 'Manual Review': '#F59E0B'}
ax = sns.countplot(
    x='class_label',
    data=train_derived,
    palette=class_colors,
    order=['Valid Claim', 'Invalid Claim', 'Manual Review']
)
plt.title("AssureX Training Set Class Distribution (N=1,050)", fontsize=14, weight='bold', pad=15)
plt.xlabel("Warranty Claim Decision Class", fontsize=12)
plt.ylabel("Number of Claim Records", fontsize=12)

n_total = len(train_derived)
for p in ax.patches:
    cnt = int(p.get_height())
    pct = (cnt / n_total) * 100
    ax.annotate(f"{cnt}\n({pct:.1f}%)",
                (p.get_x() + p.get_width() / 2., cnt / 2),
                ha='center', va='center', fontsize=12, color='white', weight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Correlation Heatmap of Numeric & Derived Features vs. Target Classes
corr_data = train_derived.copy()

# Create binary indicator flags for each target class to observe direct correlations
corr_data['target_is_valid'] = (corr_data['class_label'] == 'Valid Claim').astype(int)
corr_data['target_is_invalid'] = (corr_data['class_label'] == 'Invalid Claim').astype(int)
corr_data['target_is_manual_review'] = (corr_data['class_label'] == 'Manual Review').astype(int)

# Convert boolean flags to integers for correlation analysis
bool_cols = ['has_receipt', 'has_warranty_card', 'has_product_image', 'has_serial_evidence',
             'prior_replacement', 'has_any_contradiction']
for b in bool_cols:
    corr_data[b] = corr_data[b].astype(int)

numeric_features_for_corr = [
    'product_age_days',
    'remaining_warranty_days',
    'missing_document_count',
    'days_to_reporting_deadline',
    'repair_count',
    'has_any_contradiction',
    'purchase_price',
    'warranty_duration_months',
    'has_receipt',
    'has_warranty_card',
    'has_product_image',
    'has_serial_evidence',
    'prior_replacement',
    'target_is_valid',
    'target_is_invalid',
    'target_is_manual_review'
]

corr_matrix = corr_data[numeric_features_for_corr].corr()

plt.figure(figsize=(15, 11))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1.0,
    vmax=1.0,
    linewidths=0.5,
    cbar_kws={'label': 'Pearson Correlation Coefficient'}
)
plt.title("Correlation Heatmap: Numeric & Derived Features vs. Target Claim Decisions", fontsize=15, weight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Boxplot of Product Age (Days) by Target Class
plt.figure(figsize=(9, 6))
ax = sns.boxplot(
    x='class_label',
    y='product_age_days',
    data=train_derived,
    palette=class_colors,
    order=['Valid Claim', 'Invalid Claim', 'Manual Review'],
    showmeans=True,
    meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black", "markersize":"8"}
)

plt.title("Product Age Distribution (Days) Across Decision Classes", fontsize=14, weight='bold', pad=15)
plt.xlabel("Decision Class", fontsize=12)
plt.ylabel("Product Age in Days (Purchase to Claim Date)", fontsize=12)

# Display median annotations above boxes
class_medians = train_derived.groupby('class_label')['product_age_days'].median()
for i, cls in enumerate(['Valid Claim', 'Invalid Claim', 'Manual Review']):
    med_val = class_medians[cls]
    ax.text(
        i, med_val + 20,
        f"Median: {med_val:.0f} d",
        ha='center', va='bottom',
        fontsize=10, weight='bold',
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.85)
    )

plt.tight_layout()
plt.show()

---  
## Step 4: Categorical Encoding Strategy & Justification

### Rationale for Selected Encodings

A machine learning model requires numeric inputs, but categorical variables in the warranty claims domain exhibit drastically different cardinality levels and business semantics:

1. **`product_category` (One-Hot Encoding):**
   - **Cardinality:** Low ($K = 5$ categories: `Smartphone`, `Laptop`, `Washing Machine`, `Smart TV`, `Audio / Soundbar`).
   - **Encoding Choice:** **One-Hot Encoding** creates 5 orthogonal binary indicator columns (`category_Smartphone`, `category_Laptop`, etc.).
   - **Why It Is Optimal:** Because cardinality is small, One-Hot Encoding does not cause dimensional explosion or sparsity issues. Crucially, it avoids imposing an arbitrary ordinal ranking (e.g. `Laptop = 1, Smartphone = 2`) that would mislead linear models, distance-based algorithms, or tree split criteria.

2. **`damage_type` (Frequency & Out-of-Fold Target Encoding):**
   - **Cardinality:** High ($K = 35$ distinct manufacturing failures and exclusion modes).
   - **Encoding Choice:** **Frequency Encoding** (`damage_type_freq`) combined with **Target Risk Encoding** (`damage_type_target_invalid_prob`).
   - **Why It Is Optimal:** One-Hot Encoding on 35 categories would generate 35 highly sparse binary features, leading to tree fragmentation, excessive memory consumption, and the curse of dimensionality. Frequency encoding captures the statistical commonality of failure modes, while target encoding maps each defect directly to the empirical conditional probability $P(\text{Invalid Claim} \mid \text{damage\_type})$, capturing risk severity compactly.

3. **`retailer` (Frequency Encoding with Missing Category Indicator):**
   - **Cardinality:** Moderate ($K = 10$ authorized dealer networks + missing values).
   - **Encoding Choice:** **Frequency Encoding** (`retailer_freq`) paired with a missing retailer indicator (`retailer_is_missing`).
   - **Why It Is Optimal:** Retailer distribution reflects market penetration and dealer tier. Rather than 10 separate one-hot features, frequency encoding represents the market footprint of the seller. Missing retailer information (which strongly flags un-invoiced grey imports) is captured cleanly as an explicit feature without imputing synthetic retailer names.

### Data Leakage Mitigation Protocol
> **CRITICAL:** To prevent data leakage, **all categorical encoders (frequency dictionaries, target conditional probabilities, category levels) are fitted STRICTLY on the training split (`train_derived`)** and then applied deterministically to the validation and test splits. Unseen categories in validation/test default to the global training baseline.

In [ ]:
# ==============================================================================
# Step 4.1: Fitting Categorical Encoders on Training Split Only
# ==============================================================================

# 1. One-Hot Encoding for product_category (fitted on train)
known_categories = sorted(train_derived['product_category'].dropna().unique())
print(f"[+] Fitting One-Hot on product_category: {known_categories}")

# 2. Frequency Encoding for damage_type (fitted on train)
damage_type_freq_map = (train_derived['damage_type'].value_counts() / len(train_derived)).to_dict()

# Target Encoding for damage_type: Empirical probability of Invalid Claim (fitted on train)
global_invalid_prior = (train_derived['class_label'] == 'Invalid Claim').mean()
damage_type_invalid_prob_map = (
    train_derived.groupby('damage_type')
    .apply(lambda g: (g['class_label'] == 'Invalid Claim').sum() / len(g))
    .to_dict()
)

# 3. Frequency Encoding for retailer (fitted on train)
retailer_freq_map = (train_derived['retailer'].value_counts(dropna=True) / len(train_derived)).to_dict()

# ==============================================================================
# Step 4.2: Applying Encoders to Train, Val, and Test Splits
# ==============================================================================
def apply_categorical_encodings(
    df: pd.DataFrame,
    cat_list: list,
    dmg_freq: dict,
    dmg_prob: dict,
    ret_freq: dict,
    prior_prob: float
) -> pd.DataFrame:
    df = df.copy()

    # 1. One-Hot Encode product_category
    for cat in cat_list:
        col_name = f"category_{re.sub(r'[^a-zA-Z0-9]+', '_', cat)}"
        df[col_name] = (df['product_category'] == cat).astype(int)

    # 2. Encode damage_type
    df['damage_type_freq'] = df['damage_type'].map(dmg_freq).fillna(0.0)
    df['damage_type_invalid_risk'] = df['damage_type'].map(dmg_prob).fillna(prior_prob)

    # 3. Encode retailer
    df['retailer_freq'] = df['retailer'].map(ret_freq).fillna(0.0)
    df['retailer_is_missing'] = df['retailer'].isna().astype(int)

    return df

train_encoded = apply_categorical_encodings(
    train_derived, known_categories, damage_type_freq_map, damage_type_invalid_prob_map, retailer_freq_map, global_invalid_prior
)
val_encoded = apply_categorical_encodings(
    val_derived, known_categories, damage_type_freq_map, damage_type_invalid_prob_map, retailer_freq_map, global_invalid_prior
)
test_encoded = apply_categorical_encodings(
    test_derived, known_categories, damage_type_freq_map, damage_type_invalid_prob_map, retailer_freq_map, global_invalid_prior
)

print("[+] Categorical encodings successfully transformed across train/val/test splits!")

---  
## Step 5: Final Feature Selection & Feature Matrix Export

We assemble the finalized machine-learning-ready feature matrices:
- **Identifier:** `claim_id`
- **Target Label:** `class_label`
- **Continuous & Integer Features:** `purchase_price`, `warranty_duration_months`, `product_age_days`, `remaining_warranty_days`, `missing_document_count`, `days_to_reporting_deadline`, `repair_count`
- **Binary Evidentiary Features:** `has_receipt`, `has_warranty_card`, `has_product_image`, `has_serial_evidence`, `prior_replacement`, `has_any_contradiction`
- **Encoded Categorical Features:** `category_*` (One-Hot), `damage_type_freq`, `damage_type_invalid_risk`, `retailer_freq`, `retailer_is_missing`

We save the outputs as `train_features.csv`, `val_features.csv`, and `test_features.csv`.

In [ ]:
# Define final columns for modeling
one_hot_cols = [c for c in train_encoded.columns if c.startswith("category_")]

feature_columns = [
    'claim_id',
    'purchase_price',
    'warranty_duration_months',
    'product_age_days',
    'remaining_warranty_days',
    'missing_document_count',
    'days_to_reporting_deadline',
    'repair_count',
    'has_receipt',
    'has_warranty_card',
    'has_product_image',
    'has_serial_evidence',
    'prior_replacement',
    'has_any_contradiction',
    'damage_type_freq',
    'damage_type_invalid_risk',
    'retailer_freq',
    'retailer_is_missing',
    *one_hot_cols,
    'class_label'
]

train_features = train_encoded[feature_columns].copy()
val_features = val_encoded[feature_columns].copy()
test_features = test_encoded[feature_columns].copy()

# Convert boolean columns to integer 0/1
bool_feature_cols = ['has_receipt', 'has_warranty_card', 'has_product_image', 'has_serial_evidence',
                     'prior_replacement', 'has_any_contradiction']
for b in bool_feature_cols:
    train_features[b] = train_features[b].astype(int)
    val_features[b] = val_features[b].astype(int)
    test_features[b] = test_features[b].astype(int)

# Ensure output directories exist
for out_dir in [".", "dataset"]:
    os.makedirs(out_dir, exist_ok=True)
    train_features.to_csv(os.path.join(out_dir, "train_features.csv"), index=False)
    val_features.to_csv(os.path.join(out_dir, "val_features.csv"), index=False)
    test_features.to_csv(os.path.join(out_dir, "test_features.csv"), index=False)

print("[SUCCESS] Feature matrices saved successfully!")
print(f"  - train_features.csv: {train_features.shape[0]} rows, {train_features.shape[1]} cols")
print(f"  - val_features.csv:   {val_features.shape[0]} rows, {val_features.shape[1]} cols")
print(f"  - test_features.csv:  {test_features.shape[0]} rows, {test_features.shape[1]} cols")

print("\nSample features preview:")
train_features.head()